In [0]:
%pip install databricks-labs-dqx

In [0]:
dbutils.library.restartPython()

In [0]:
from databricks.labs.dqx.engine import DQEngine
print("DQX imported successfully")

In [0]:
# Record the DQX version, since its check arguments have changed between releases
import importlib.metadata
print(importlib.metadata.version("databricks-labs-dqx"))

In [0]:
# Create the DQX engine and define uniqueness and not-null checks for c_custkey
from databricks.labs.dqx.engine import DQEngine
from databricks.sdk import WorkspaceClient

dq_engine = DQEngine(WorkspaceClient())

checks = [
    {"criticality": "error",
     "check": {"function": "is_unique", "arguments": {"columns": ["c_custkey"]}}},
    {"criticality": "error",
     "check": {"function": "is_not_null", "for_each_column": ["c_custkey"]}},
]

In [0]:
# Apply the checks to a clean table and a table with planted duplicates
for table_name in ["validation_sweep.dev.customer", "validation_sweep.dev.customer_dupes"]:
    df = spark.table(table_name)
    valid_df, invalid_df = dq_engine.apply_checks_by_metadata_and_split(df, checks)
    print(f"{table_name:40} valid rows: {valid_df.count():>8,}  invalid rows: {invalid_df.count():>4,}")

In [0]:
display(invalid_df)

In [0]:
# Apply composite-key checks to the table with planted null key values
partsupp_checks = [
    {"criticality": "error",
     "check": {"function": "is_unique", "arguments": {"columns": ["ps_partkey", "ps_suppkey"]}}},
    {"criticality": "error",
     "check": {"function": "is_not_null", "for_each_column": ["ps_partkey", "ps_suppkey"]}},
]

for table_name in ["validation_sweep.dev.partsupp", "validation_sweep.dev.partsupp_nulls"]:
    df = spark.table(table_name)
    valid_df, invalid_df = dq_engine.apply_checks_by_metadata_and_split(df, partsupp_checks)
    print(f"{table_name:40} valid rows: {valid_df.count():>8,}  invalid rows: {invalid_df.count():>4,}")

In [0]:
# Convert DQX's duplicate flags into extra-row counts, matching the results table definition
from pyspark.sql import functions as F

df = spark.table("validation_sweep.dev.customer_dupes")
valid_df, invalid_df = dq_engine.apply_checks_by_metadata_and_split(df, checks)

flagged_rows = invalid_df.count()
flagged_keys = invalid_df.select("c_custkey").distinct().count()
print(f"flagged rows: {flagged_rows}, distinct keys: {flagged_keys}, extra rows: {flagged_rows - flagged_keys}")